In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import text, create_engine
from electricity_forecast.config import settings
from electricity_forecast.features import add_price_lags, add_cyclical_hour, add_cyclical_dayofweek

engine = create_engine(settings.database_url)

In [ ]:
query_pvpc = text("""
    SELECT
        price_datetime AT TIME ZONE 'Europe/Madrid' AS price_datetime_local,
        price_eur_mwh
    FROM pvpc_prices
    ORDER BY price_datetime
""")
df_pvpc_historico = pd.read_sql( query_pvpc, engine )

query_spot = text("""
    SELECT
        price_datetime AT TIME ZONE 'Europe/Madrid' AS price_datetime_local,
        price_eur_mwh
    FROM spot_market_prices
    ORDER BY price_datetime
""")
df_spot_historico = pd.read_sql( query_spot, engine )

df_pvpc_historico.head()
df_pvpc_historico.info()

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(df_pvpc_historico["price_datetime_local"], df_pvpc_historico["price_eur_mwh"])
plt.title("PVPC - Histórico completo")
plt.xlabel("Fecha")
plt.ylabel("€/MWh")
plt.show()

In [ ]:
df_pvpc_historico["price_eur_mwh"].describe()


In [ ]:
(df_pvpc_historico["price_eur_mwh"] < 0).sum()

In [ ]:
df_pvpc_historico["hour"] = df_pvpc_historico["price_datetime_local"].dt.hour

media_precio_por_hora = (
    df_pvpc_historico.groupby("hour")["price_eur_mwh"]
    .mean()
    .sort_index()
)

media_precio_por_hora.plot(kind="bar", figsize=(12, 5))
plt.title("Precio medio por hora - PVPC")
plt.xlabel("Hora del día")
plt.ylabel("Precio medio (€/MWh)")
plt.xticks(rotation=0)
plt.show()

In [ ]:
df_pvpc_historico["tipo_dia"] = df_pvpc_historico["price_datetime_local"].dt.dayofweek.map(
    lambda dia: "Día laboral" if dia < 5 else "Fin de semana"
)

df_pvpc_historico["tipo_dia"] = pd.Categorical(
    df_pvpc_historico["tipo_dia"],
    categories=["Día laboral", "Fin de semana"],
    ordered=True,
)

media_precio_por_tipo_dia = (
    df_pvpc_historico.groupby("tipo_dia", observed=False)["price_eur_mwh"]
    .mean()
)

media_precio_por_tipo_dia.plot(kind="bar", figsize=(8, 5))
plt.title("Precio medio: día laboral vs fin de semana")
plt.xlabel("Tipo de día")
plt.ylabel("Precio medio (€/MWh)")
plt.xticks(rotation=0)
plt.show()

In [ ]:
df_spot_features = df_spot_historico.set_index("price_datetime_local")
df_spot_features = add_price_lags(df_spot_features, [24])
df_spot_features[["price_eur_mwh", "precio_lag_hours_24"]].head(10)

In [ ]:
df_spot_features = df_spot_historico.set_index("price_datetime_local")
df_spot_features = add_price_lags(df_spot_features, [24, 48, 168])
df_spot_features.info()

In [ ]:
df_spot_features = add_cyclical_hour(df_spot_features)

plt.figure(figsize=(6, 6))
plt.scatter(df_spot_features["hora_sin"], df_spot_features["hora_cos"], s=5)
plt.xlabel("hora_sin")
plt.ylabel("hora_cos")
plt.title("Codificación cíclica de la hora")
plt.axis("equal")
plt.show()

In [ ]:
df_spot_features.loc["2026-09-01 23:45:00", ["hora_sin", "hora_cos"]]


In [ ]:
df_spot_features.loc["2026-09-01 00:00:00", ["hora_sin", "hora_cos"]]

In [ ]:
df_spot_features = add_cyclical_dayofweek(df_spot_features)
df_spot_features.loc["2026-09-06 23:45:00", ["day_sin", "day_cos"]]   # domingo, última hora del día



In [ ]:
df_spot_features.loc["2026-09-07 00:00:00", ["day_sin", "day_cos"]]   # lunes, primera hora del día

In [ ]:
from electricity_forecast.weather_client import fetch_hourly_forecast
from electricity_forecast.config import settings

forecast = fetch_hourly_forecast(settings.aemet_municipio_id, settings.aemet_api_key)
forecast[0].keys()

In [ ]:
import requests

fecha_ini = "2026-09-01T00:00:00UTC"
fecha_fin = "2026-09-03T23:59:59UTC"
estacion_id = "3182Y"

url = f"https://opendata.aemet.es/opendata/api/valores/climatologicos/diarios/datos/fechaini/{fecha_ini}/fechafin/{fecha_fin}/estacion/{estacion_id}"

response = requests.get(url, headers={"api_key": settings.aemet_api_key})
response.raise_for_status()
datos_url = response.json()["datos"]

response_datos = requests.get(datos_url)
response_datos.encoding = "ISO-8859-1"
historico = response_datos.json()

print(type(historico))
print(len(historico))
print(historico[0])

In [ ]:
from electricity_forecast.transform import parse_weather_records

registros = parse_weather_records(historico, is_real=True)
registros

In [ ]:
from sqlalchemy.orm import Session
from electricity_forecast.load import load_weather_records

with Session(engine) as session:
    load_weather_records(registros, session)

In [ ]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT * FROM weather_records"))
    for row in result:
        print(row)

In [ ]:
fecha_ini = "2026-09-01T00:00:00UTC"
fecha_fin = "2026-09-23T23:59:59UTC"
estacion_id = "3182Y"

url = f"https://opendata.aemet.es/opendata/api/valores/climatologicos/diarios/datos/fechaini/{fecha_ini}/fechafin/{fecha_fin}/estacion/{estacion_id}"

response = requests.get(url, headers={"api_key": settings.aemet_api_key})
response.raise_for_status()
datos_url = response.json()["datos"]

response_datos = requests.get(datos_url)
response_datos.encoding = "ISO-8859-1"
historico_completo = response_datos.json()

fechas_recibidas = {item["fecha"] for item in historico_completo}
fechas_esperadas = {f"2026-09-{dia:02d}" for dia in range(1, 24)}

fechas_faltantes = fechas_esperadas - fechas_recibidas

for item in historico_completo:
    if "tmed" not in item:
        print(item)

In [ ]:
from electricity_forecast.transform import parse_weather_records
from electricity_forecast.load import load_weather_records

registros_completos = parse_weather_records(historico_completo, is_real=True)

with Session(engine) as session:
    load_weather_records(registros_completos, session)

with engine.connect() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM weather_records"))
    print(result.scalar())